In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Definición de rutas de ficheros
export_ine_path = Path(r"C:\Users\LorenaMéndez\Desktop\Analisis_Precios_vivienda\Export_INE")
ipv_25171_file_path = export_ine_path / "IPV - Export 25171.xlsx"

In [3]:
# Cargar fichero de exportación del indice del IPV
df_ipv_25171 = pd.read_excel(ipv_25171_file_path, sheet_name="tabla-25171-indice")

# Usar la primera fila como nombres de columnas
df_ipv_25171.columns = df_ipv_25171.iloc[0]
# Eliminar la primera fila del DataFrame
df_ipv_25171 = df_ipv_25171.iloc[1:]
# Resetear índice
df_ipv_25171 = df_ipv_25171.reset_index(drop=True)

# Generar columnas de agrupación por región geográfica
df_ipv_25171_agrup = (
    df_ipv_25171
        .rename(columns={df_ipv_25171.columns[0]: "detalle"})
        .assign(
            ref_geografica=lambda d: d["detalle"]
                .where(d.drop(columns="detalle").isna().all(axis=1))
                .ffill(),
            tipo_indice=lambda d: d["detalle"]
                .where(~d.drop(columns="detalle").isna().all(axis=1))
        )
)

df_ipv_25171_agrup = df_ipv_25171_agrup[~df_ipv_25171_agrup.drop(columns=["detalle", "ref_geografica", "tipo_indice"])
        .isna().all(axis=1)]

df_ipv_25171_agrup = df_ipv_25171_agrup.drop(columns="detalle").reset_index(drop=True)

# Trasponer columnas de fechas por filas
df_ipv_25171_agrup = df_ipv_25171_agrup.melt(
    id_vars=["ref_geografica", "tipo_indice"],
    var_name="periodo",
    value_name="valor"
)

# Eliminar valores en blanco al inicio de columnas
df_ipv_25171_agrup["tipo_indice"] = df_ipv_25171_agrup["tipo_indice"].str.lstrip()
df_ipv_25171_agrup["ref_geografica"] = df_ipv_25171_agrup["ref_geografica"].str.lstrip()

# Añadir nuevas columnas de tiempo
df_ipv_25171_agrup["ejercicio"] =  df_ipv_25171_agrup["periodo"].str[:4].astype(int)
df_ipv_25171_agrup["trimestre"] =  df_ipv_25171_agrup["periodo"].str[-2:]

# Añadir columna de fecha
periodo_q = df_ipv_25171_agrup["periodo"].str.replace("T", "Q", regex=False)

df_ipv_25171_agrup["fecha"] = (
    pd.PeriodIndex(periodo_q, freq="Q")
      .to_timestamp(how="start")
)

# Reordenar columnas
df_ipv_25171_agrup = df_ipv_25171_agrup[
    ["ejercicio", "trimestre", "fecha",
        "ref_geografica", "tipo_indice"] +
    df_ipv_25171_agrup.columns.drop(["ref_geografica", "tipo_indice","ejercicio", "trimestre", "fecha"]).tolist()
]

# Visualizar resultados del IPV
df_ipv_25171_agrup.head()

,ejercicio,trimestre,fecha,ref_geografica,tipo_indice,periodo,valor
0,2025,T3,2025-07-01,Nacional,General,2025T3,183.389
1,2025,T3,2025-07-01,Nacional,Vivienda nueva,2025T3,207.214
2,2025,T3,2025-07-01,Nacional,Vivienda segunda mano,2025T3,179.657
3,2025,T3,2025-07-01,01 Andalucía,General,2025T3,182.372
4,2025,T3,2025-07-01,01 Andalucía,Vivienda nueva,2025T3,221.148


In [ ]:
# Exportar los resultados a excel
df_ipv_25171_agrup.to_excel("ipv_25171_agrup.xlsx", index=False)

In [ ]:
# Mover el fichero al directorio de destino

# carpeta de origen
origen = Path("ipv_25171_agrup.xlsx")

# carpeta destino
output_dir = Path(
    r"C:\Users\LorenaMéndez\Desktop\Analisis_Precios_vivienda\Datos INE Procesados"
)
output_dir.mkdir(parents=True, exist_ok=True)

# ruta completa de destino
destino = output_dir / origen.name

# si existe en destino, lo eliminamos (para permitir sobrescritura)
if destino.exists():
    destino.unlink()

# mover el fichero (esto elimina el original de la raíz)
origen.rename(destino)

In [4]:
# Cargar fichero de exportación de variación trimestral del indice del IPV
df_ipv_25171_var_trim = pd.read_excel(ipv_25171_file_path, sheet_name="tabla-25171-vartrim")

# Usar la primera fila como nombres de columnas
df_ipv_25171_var_trim.columns = df_ipv_25171_var_trim.iloc[0]
# Eliminar la primera fila del DataFrame
df_ipv_25171_var_trim = df_ipv_25171_var_trim.iloc[1:]
# Resetear índice
df_ipv_25171_var_trim = df_ipv_25171_var_trim.reset_index(drop=True)

# Generar columnas de agrupación por región geográfica
df_ipv_25171_var_trim_agrup = (
    df_ipv_25171_var_trim
        .rename(columns={df_ipv_25171_var_trim.columns[0]: "detalle"})
        .assign(
            ref_geografica=lambda d: d["detalle"]
                .where(d.drop(columns="detalle").isna().all(axis=1))
                .ffill(),
            tipo_indice=lambda d: d["detalle"]
                .where(~d.drop(columns="detalle").isna().all(axis=1))
        )
)

df_ipv_25171_var_trim_agrup = df_ipv_25171_var_trim_agrup[~df_ipv_25171_var_trim_agrup.drop(columns=["detalle", "ref_geografica", "tipo_indice"])
        .isna().all(axis=1)]

df_ipv_25171_var_trim_agrup = df_ipv_25171_var_trim_agrup.drop(columns="detalle").reset_index(drop=True)

# Trasponer columnas de fechas por filas
df_ipv_25171_var_trim_agrup = df_ipv_25171_var_trim_agrup.melt(
    id_vars=["ref_geografica", "tipo_indice"],
    var_name="periodo",
    value_name="valor"
)

# Eliminar valores en blanco al inicio de columnas
df_ipv_25171_var_trim_agrup["tipo_indice"] = df_ipv_25171_var_trim_agrup["tipo_indice"].str.lstrip()
df_ipv_25171_var_trim_agrup["ref_geografica"] = df_ipv_25171_var_trim_agrup["ref_geografica"].str.lstrip()

# Añadir nuevas columnas de tiempo
df_ipv_25171_var_trim_agrup["ejercicio"] =  df_ipv_25171_var_trim_agrup["periodo"].str[:4].astype(int)
df_ipv_25171_var_trim_agrup["trimestre"] =  df_ipv_25171_var_trim_agrup["periodo"].str[-2:]

# Añadir columna de fecha
periodo_q = df_ipv_25171_var_trim_agrup["periodo"].str.replace("T", "Q", regex=False)

df_ipv_25171_var_trim_agrup["fecha"] = (
    pd.PeriodIndex(periodo_q, freq="Q")
      .to_timestamp(how="start")
)

# Reordenar columnas
df_ipv_25171_var_trim_agrup = df_ipv_25171_var_trim_agrup[
    ["ejercicio", "trimestre", "fecha",
        "ref_geografica", "tipo_indice"] +
    df_ipv_25171_var_trim_agrup.columns.drop(["ref_geografica", "tipo_indice","ejercicio", "trimestre", "fecha"]).tolist()
]

# Visualizar resultados del IPV
df_ipv_25171_var_trim_agrup.head()

,ejercicio,trimestre,fecha,ref_geografica,tipo_indice,periodo,valor
0,2025,T3,2025-07-01,Nacional,General,2025T3,2.9
1,2025,T3,2025-07-01,Nacional,Vivienda nueva,2025T3,0.6
2,2025,T3,2025-07-01,Nacional,Vivienda segunda mano,2025T3,3.3
3,2025,T3,2025-07-01,01 Andalucía,General,2025T3,2.7
4,2025,T3,2025-07-01,01 Andalucía,Vivienda nueva,2025T3,1.3


In [ ]:
# Exportar los resultados a excel
df_ipv_25171_var_trim_agrup.to_excel("ipv_25171_var_trim_agrup.xlsx", index=False)

In [ ]:
# Mover el fichero al directorio de destino

# carpeta de origen
origen = Path("ipv_25171_var_trim_agrup.xlsx")

# carpeta destino
output_dir = Path(
    r"C:\Users\LorenaMéndez\Desktop\Analisis_Precios_vivienda\Datos INE Procesados"
)
output_dir.mkdir(parents=True, exist_ok=True)

# ruta completa de destino
destino = output_dir / origen.name

# si existe en destino, lo eliminamos (para permitir sobrescritura)
if destino.exists():
    destino.unlink()

# mover el fichero (esto elimina el original de la raíz)
origen.rename(destino)

In [9]:
# Cargar fichero de exportación de variación trimestral del indice del IPV
df_ipv_25171_var_anual = pd.read_excel(ipv_25171_file_path, sheet_name="tabla-25171-varanual")

# Usar la primera fila como nombres de columnas
df_ipv_25171_var_anual.columns = df_ipv_25171_var_anual.iloc[0]
# Eliminar la primera fila del DataFrame
df_ipv_25171_var_anual = df_ipv_25171_var_anual.iloc[1:]
# Resetear índice
df_ipv_25171_var_anual = df_ipv_25171_var_anual.reset_index(drop=True)

# Generar columnas de agrupación por región geográfica
df_ipv_25171_var_anual_agrup = (
    df_ipv_25171_var_anual
        .rename(columns={df_ipv_25171_var_anual.columns[0]: "detalle"})
        .assign(
            ref_geografica=lambda d: d["detalle"]
                .where(d.drop(columns="detalle").isna().all(axis=1))
                .ffill(),
            tipo_indice=lambda d: d["detalle"]
                .where(~d.drop(columns="detalle").isna().all(axis=1))
        )
)

df_ipv_25171_var_anual_agrup = df_ipv_25171_var_anual_agrup[~df_ipv_25171_var_anual_agrup.drop(columns=["detalle", "ref_geografica", "tipo_indice"])
        .isna().all(axis=1)]

df_ipv_25171_var_anual_agrup = df_ipv_25171_var_anual_agrup.drop(columns="detalle").reset_index(drop=True)

# Trasponer columnas de fechas por filas
df_ipv_25171_var_anual_agrup = df_ipv_25171_var_anual_agrup.melt(
    id_vars=["ref_geografica", "tipo_indice"],
    var_name="periodo",
    value_name="valor"
)

# Eliminar valores en blanco al inicio de columnas
df_ipv_25171_var_anual_agrup["tipo_indice"] = df_ipv_25171_var_anual_agrup["tipo_indice"].str.lstrip()
df_ipv_25171_var_anual_agrup["ref_geografica"] = df_ipv_25171_var_anual_agrup["ref_geografica"].str.lstrip()

# Añadir nuevas columnas de tiempo
df_ipv_25171_var_anual_agrup["ejercicio"] =  df_ipv_25171_var_anual_agrup["periodo"].str[:4].astype(int)
df_ipv_25171_var_anual_agrup["trimestre"] =  df_ipv_25171_var_anual_agrup["periodo"].str[-2:]

# Añadir columna de fecha
periodo_q = df_ipv_25171_var_anual_agrup["periodo"].str.replace("T", "Q", regex=False)

df_ipv_25171_var_anual_agrup["fecha"] = (
    pd.PeriodIndex(periodo_q, freq="Q")
      .to_timestamp(how="start")
)

# Reordenar columnas
df_ipv_25171_var_anual_agrup = df_ipv_25171_var_anual_agrup[
    ["ejercicio", "trimestre", "fecha",
        "ref_geografica", "tipo_indice"] +
    df_ipv_25171_var_anual_agrup.columns.drop(["ref_geografica", "tipo_indice","ejercicio", "trimestre", "fecha"]).tolist()
]

# Visualizar resultados del IPV
df_ipv_25171_var_anual_agrup.head()

,ejercicio,trimestre,fecha,ref_geografica,tipo_indice,periodo,valor
0,2025,T3,2025-07-01,Nacional,General,2025T3,12.8
1,2025,T3,2025-07-01,Nacional,Vivienda nueva,2025T3,9.7
2,2025,T3,2025-07-01,Nacional,Vivienda segunda mano,2025T3,13.4
3,2025,T3,2025-07-01,01 Andalucía,General,2025T3,12.4
4,2025,T3,2025-07-01,01 Andalucía,Vivienda nueva,2025T3,9.7


In [10]:
# Exportar los resultados a excel
df_ipv_25171_var_anual_agrup.to_excel("ipv_25171_var_anual_agrup.xlsx", index=False)

In [11]:
# Mover el fichero al directorio de destino

# carpeta de origen
origen = Path("ipv_25171_var_anual_agrup.xlsx")

# carpeta destino
output_dir = Path(
    r"C:\Users\LorenaMéndez\Desktop\Analisis_Precios_vivienda\Datos INE Procesados"
)
output_dir.mkdir(parents=True, exist_ok=True)

# ruta completa de destino
destino = output_dir / origen.name

# si existe en destino, lo eliminamos (para permitir sobrescritura)
if destino.exists():
    destino.unlink()

# mover el fichero (esto elimina el original de la raíz)
origen.rename(destino)

WindowsPath('C:/Users/LorenaMéndez/Desktop/Analisis_Precios_vivienda/Datos INE Procesados/ipv_25171_var_anual_agrup.xlsx')